## Homework

The goal of this homework is to get familiar with MLflow, the tool for experiment tracking and 
model management.

## Q1. Install MLflow

To get started with MLflow you'll need to install the MLflow Python package.

For this we recommend creating a separate Python environment, for example, you can use [conda environments](https://docs.conda.io/projects/conda/en/latest/user-guide/getting-started.html#managing-envs), and then install the package there with `pip` or `conda`.

Once you installed the package, run the command `mlflow --version` and check the output.

What's the version that you have?

In [1]:
import mlflow

print('MLflow version:', mlflow.__version__)

MLflow version: 3.17.2


**Answer:** 3.17.2

## Q2. Download and preprocess the data

We'll use the Green Taxi Trip Records dataset to predict the duration of each trip. 

Download the data for January, February and March 2023 in parquet format from [here](https://www1.nyc.gov/site/tlc/about/tlc-trip-record-data.page).

Use the script `preprocess_data.py` located in the folder [`homework`](homework) to preprocess the data.

The script will:

* load the data from the folder `<TAXI_DATA_FOLDER>` (the folder where you have downloaded the data),
* fit a `DictVectorizer` on the training set (January 2023 data),
* save the preprocessed datasets and the `DictVectorizer` to disk.

Your task is to download the datasets and then execute this command:

```
python preprocess_data.py --raw_data_path <TAXI_DATA_FOLDER> --dest_path ./output
```
Tip: go to `02-experiment-tracking/homework/` folder before executing the command and change the value of `<TAXI_DATA_FOLDER>` to the location where you saved the data.

How many files were saved to `OUTPUT_FOLDER`?

* 1
* 3
* 4
* 7

In [2]:
%run preprocess_data.py --raw_data_path ./taxi_data_folder --dest_path ./output

In [3]:
import os

files = os.listdir('output')
print(len(files), 'files:', files)

4 files: ['train.pkl', 'val.pkl', 'dv.pkl', 'test.pkl']


**Answer:** 4

## Q3. Train a model with autolog

We will train a `RandomForestRegressor` (from Scikit-Learn) on the taxi dataset.

We have prepared the training script `train.py` for this exercise, which can be also found in the folder [`homework`](homework). 

The script will:

* load the datasets produced by the previous step,
* train the model on the training set,
* calculate the RMSE score on the validation set.

Your task is to modify the script to enable **autologging** with MLflow, execute the script and then launch the MLflow UI to check that the experiment run was properly tracked. 

Tip 1: don't forget to wrap the training code with a `with mlflow.start_run():` statement as we showed in the videos.

Tip 2: don't modify the hyperparameters of the model to make sure that the training will finish quickly.

What is the value of the `min_samples_split` parameter:

* 2
* 4
* 8
* 10

In [4]:
# train.py calls mlflow.sklearn.autolog() and trains inside `with mlflow.start_run():`
# it logs to the local file mlflow.db
%run train.py

2026/10/26 21:28:22 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/26 21:28:22 INFO mlflow.store.db.utils: Updating database tables
2026/10/26 21:28:23 INFO mlflow.tracking.fluent: Experiment with name 'nyc-taxi-experiment' does not exist. Creating a new experiment.
2026/10/26 21:28:23 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 1.6.0 <= scikit-learn, but the installed version is 1.5.0. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
2026/10/26 21:28:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persi

validation rmse: 5.431160019389339
min_samples_split: 2


In [5]:
from mlflow.tracking import MlflowClient

client = MlflowClient('sqlite:///mlflow.db')
exp = client.get_experiment_by_name('nyc-taxi-experiment')
run = client.search_runs([exp.experiment_id], order_by=['attributes.start_time DESC'], max_results=1)[0]
print('min_samples_split:', run.data.params['min_samples_split'])

min_samples_split: 2


**Answer:** 2

## Q4. Launch the tracking server locally

Now we want to manage the entire lifecycle of our ML model. In this step, you'll need to launch a tracking server. This way we will also have access to the model registry. 

Your task is to:

* launch the tracking server on your local machine,
* select a SQLite db for the backend store and a folder called `artifacts` for the artifacts store.

You should keep the tracking server running to work on the next two exercises that use the server.

In addition to `backend-store-uri`, what else do you need to pass to properly configure the server?

* `default-artifact-root`
* `serve-artifacts`
* `artifacts-only`
* `artifacts-destination`

**Answer:** default-artifact-root

## Q5. Tune model hyperparameters

Now let's try to reduce the validation error by tuning the hyperparameters of the `RandomForestRegressor` using `hyperopt`. 
We have prepared the script `hpo.py` for this exercise. 

Your task is to modify the script `hpo.py` and make sure that the validation RMSE is logged to the tracking server for each run of the hyperparameter optimization (you will need to add a few lines of code to the `objective` function) and run the script without passing any parameters.

After that, open UI and explore the runs from the experiment called `random-forest-hyperopt` to answer the question below.

Note: Don't use autologging for this exercise.

The idea is to just log the information that you need to answer the question below, including:

* the list of hyperparameters that are passed to the `objective` function during the optimization,
* the RMSE obtained on the validation set (February 2023 data).

What's the best validation RMSE that you got?

* 4.817
* 5.335
* 5.818
* 6.336

In [6]:
# hpo.py logs the params and `rmse` of every trial inside `objective` (no autolog)
%run hpo.py

2026/10/26 21:33:46 INFO mlflow.tracking.fluent: Experiment with name 'random-forest-hyperopt' does not exist. Creating a new experiment.


  0%|          | 0/15 [00:00<?, ?trial/s, best loss=?]

2026/10/26 21:33:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



🏃 View run brawny-squirrel-389 at: http://127.0.0.1:5001/#/experiments/1/runs/9391d27ce73c46d68c201cbb19851d89

🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1

  7%|▋         | 1/15 [00:12<02:52, 12.33s/trial, best loss: 5.370086069268862]

2026/10/26 21:34:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



...

🏃 View run nimble-asp-230 at: http://127.0.0.1:5001/#/experiments/1/runs/6ebf3483a7db4bc6abbafc3d2e5ee939

🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1                    

 87%|████████▋ | 13/15 [01:50<00:15,  7.68s/trial, best loss: 5.335419588556921]

2026/10/26 21:35:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



🏃 View run adaptable-elk-844 at: http://127.0.0.1:5001/#/experiments/1/runs/6d47e5152671430a8957d74a46135636

🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1                    

 93%|█████████▎| 14/15 [01:57<00:07,  7.62s/trial, best loss: 5.335419588556921]

2026/10/26 21:35:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



🏃 View run rebellious-skink-596 at: http://127.0.0.1:5001/#/experiments/1/runs/32bf67d3b8ff4a43b54c0b4b9525f3f2

🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1                    

100%|██████████| 15/15 [02:07<00:00,  8.47s/trial, best loss: 5.335419588556921]


In [7]:
client = MlflowClient('http://127.0.0.1:5001')
exp = client.get_experiment_by_name('random-forest-hyperopt')
best = client.search_runs([exp.experiment_id], order_by=['metrics.rmse ASC'], max_results=1)[0]
print('best validation RMSE:', best.data.metrics['rmse'])
print('params:', best.data.params)

best validation RMSE: 5.335419588556921
params: {'max_depth': '19', 'min_samples_leaf': '2', 'min_samples_split': '2', 'n_estimators': '11', 'random_state': '42', 'bootstrap': 'True', 'ccp_alpha': '0.0', 'criterion': 'squared_error', 'max_features': '1.0', 'max_leaf_nodes': 'None', 'max_samples': 'None', 'min_impurity_decrease': '0.0', 'min_weight_fraction_leaf': '0.0', 'monotonic_cst': 'None', 'n_jobs': 'None', 'oob_score': 'False', 'verbose': '0', 'warm_start': 'False'}


**Answer:** 5.335

## Q6. Promote the best model to the model registry

The results from the hyperparameter optimization are quite good. So, we can assume that we are ready to test some of these models in production. 
In this exercise, you'll promote the best model to the model registry. We have prepared a script called `register_model.py`, which will check the results from the previous step and select the top 5 runs. 
After that, it will calculate the RMSE of those models on the test set (March 2023 data) and save the results to a new experiment called `random-forest-best-models`.

Your task is to update the script `register_model.py` so that it selects the model with the lowest RMSE on the test set and registers it to the model registry.

Tip 1: you can use the method `search_runs` from the `MlflowClient` to get the model with the lowest RMSE,

Tip 2: to register the model you can use the method `mlflow.register_model` and you will need to pass the right `model_uri` in the form of a string that looks like this: `"runs:/<RUN_ID>/model"`, and the name of the model (make sure to choose a good one!).

What is the test RMSE of the best model?

* 5.060
* 5.567
* 6.061
* 6.568

In [8]:
# register_model.py re-trains the top 5 runs, picks the lowest test_rmse and registers it as 'nyc-taxi-rf-best'
%run register_model.py

2026/10/26 21:37:14 INFO mlflow.tracking.fluent: Experiment with name 'random-forest-best-models' does not exist. Creating a new experiment.
2026/10/26 21:37:14 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 1.6.0 <= scikit-learn, but the installed version is 1.5.0. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
2026/10/26 21:37:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run delightful-bird-260 at: http://127.0.0.1:5001/#/experiments/2/runs/b179f4eda77545a4ba75e66190642503
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


2026/10/26 21:37:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run resilient-swan-206 at: http://127.0.0.1:5001/#/experiments/2/runs/c933ad9770174324b89852582ad32af4
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


2026/10/26 21:37:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

2026/10/26 21:37:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run painted-worm-908 at: http://127.0.0.1:5001/#/experiments/2/runs/1b04704bccf84bcc98336c6056035a4e
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


2026/10/26 21:38:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'nyc-taxi-rf-best'.
2026/10/26 21:38:08 WARNING mlflow.tracking._model_registry.fluent: Run with id b179f4eda77545a4ba75e66190642503 has no artifacts at artifact path 'model', registering model based on models:/m-2f4d8e633c4c4633a2ed5400f6c71802 instead
2026/10/26 21:38:08 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: nyc-taxi-rf-best, version 1


🏃 View run gentle-midge-394 at: http://127.0.0.1:5001/#/experiments/2/runs/ef4176d73f584a1c9224f70e93e771c0
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2
best run: b179f4eda77545a4ba75e66190642503 test_rmse: 5.567408012462019


Created version '1' of model 'nyc-taxi-rf-best'.


In [9]:
exp = client.get_experiment_by_name('random-forest-best-models')
best = client.search_runs([exp.experiment_id], order_by=['metrics.test_rmse ASC'], max_results=1)[0]
print('test RMSE of best model:', best.data.metrics['test_rmse'])

test RMSE of best model: 5.567408012462019


**Answer:** 5.567

## Submit the results

* Submit your results here: https://courses.datatalks.club/mlops-zoomcamp-2025/homework/hw2
* If your answer doesn't match options exactly, select the closest one